# Feature Engineering

Feature engineering is the process of creating new variables and transforming existing ones to improve the predictive performance of machine learning models.

This notebook focuses on generating meaningful features from the flight dataset while preventing data leakage. Variables that would reveal the outcome of the flight after departure will be identified and excluded from the final modeling dataset.

The resulting dataset will contain only information that would realistically be available before or at the scheduled departure time.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

pd.set_option("display.max_columns", None)

In [2]:
from pathlib import Path


def find_project_root() -> Path:
    """
    Locate the project root whether the notebook is launched
    from the project root or from inside the notebooks folder.
    """
    current_path = Path.cwd().resolve()

    for candidate in [current_path, *current_path.parents]:
        if (
            (candidate / "notebooks").is_dir()
            and (candidate / "requirements.txt").is_file()
        ):
            return candidate

    raise FileNotFoundError(
        "Project root could not be located. "
        "Run this notebook from inside the "
        "Flight_Delay_Prediction project folder."
    )


PROJECT_ROOT = find_project_root()

RAW_DATA_DIRECTORY = PROJECT_ROOT / "data" / "raw"
PROCESSED_DATA_DIRECTORY = PROJECT_ROOT / "data" / "processed"
MODELS_DIRECTORY = PROJECT_ROOT / "models"

RAW_DATA_DIRECTORY.mkdir(parents=True, exist_ok=True)
PROCESSED_DATA_DIRECTORY.mkdir(parents=True, exist_ok=True)
MODELS_DIRECTORY.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)

Project root: /Users/omarzakzook/Desktop/Flight_Delay_Prediction


In [3]:
preprocessed_data_path = (
    PROCESSED_DATA_DIRECTORY
    / "preprocessed_flight_data.csv"
)

df = pd.read_csv(
    preprocessed_data_path,
    parse_dates=["FL_DATE"]
)
df.head()

,FL_DATE,AIRLINE,AIRLINE_DOT,AIRLINE_CODE,DOT_CODE,FL_NUMBER,ORIGIN,ORIGIN_CITY,DEST,DEST_CITY,CRS_DEP_TIME,DEP_TIME,DEP_DELAY,TAXI_OUT,WHEELS_OFF,WHEELS_ON,TAXI_IN,CRS_ARR_TIME,ARR_TIME,ARR_DELAY,CANCELLED,CANCELLATION_CODE,DIVERTED,CRS_ELAPSED_TIME,ELAPSED_TIME,AIR_TIME,DISTANCE,DELAY_DUE_CARRIER,DELAY_DUE_WEATHER,DELAY_DUE_NAS,DELAY_DUE_SECURITY,DELAY_DUE_LATE_AIRCRAFT,IS_DELAYED
0,2019-01-09,United Air Lines Inc.,United Air Lines Inc.: UA,UA,19977,1562,FLL,"Fort Lauderdale, FL",EWR,"Newark, NJ",1155,1151.0,-4.0,19.0,1210.0,1443.0,4.0,1501,1447.0,-14.0,0.0,NaN,0.0,186.0,176.0,153.0,1065.0,NaN,NaN,NaN,NaN,NaN,0
1,2022-11-19,Delta Air Lines Inc.,Delta Air Lines Inc.: DL,DL,19790,1149,MSP,"Minneapolis, MN",SEA,"Seattle, WA",2120,2114.0,-6.0,9.0,2123.0,2232.0,38.0,2315,2310.0,-5.0,0.0,NaN,0.0,235.0,236.0,189.0,1399.0,NaN,NaN,NaN,NaN,NaN,0
2,2022-07-22,United Air Lines Inc.,United Air Lines Inc.: UA,UA,19977,459,DEN,"Denver, CO",MSP,"Minneapolis, MN",954,1000.0,6.0,20.0,1020.0,1247.0,5.0,1252,1252.0,0.0,0.0,NaN,0.0,118.0,112.0,87.0,680.0,NaN,NaN,NaN,NaN,NaN,0
3,2023-03-06,Delta Air Lines Inc.,Delta Air Lines Inc.: DL,DL,19790,2295,MSP,"Minneapolis, MN",SFO,"San Francisco, CA",1609,1608.0,-1.0,27.0,1635.0,1844.0,9.0,1829,1853.0,24.0,0.0,NaN,0.0,260.0,285.0,249.0,1589.0,0.0,0.0,24.0,0.0,0.0,1
4,2020-02-23,Spirit Air Lines,Spirit Air Lines: NK,NK,20416,407,MCO,"Orlando, FL",DFW,"Dallas/Fort Worth, TX",1840,1838.0,-2.0,15.0,1853.0,2026.0,14.0,2041,2040.0,-1.0,0.0,NaN,0.0,181.0,182.0,153.0,985.0,NaN,NaN,NaN,NaN,NaN,0


In [4]:
print(f"Rows: {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

Rows: 2,913,802
Columns: 33


### 📝 Interpretation

The preprocessed dataset was loaded successfully. It contains the cleaned observations from the previous notebook and will serve as the foundation for feature engineering.

In [5]:
# Year
df["YEAR"] = df["FL_DATE"].dt.year

# Month
df["MONTH"] = df["FL_DATE"].dt.month

# Day
df["DAY"] = df["FL_DATE"].dt.day

# Day of week
df["DAY_OF_WEEK"] = df["FL_DATE"].dt.day_name()

# Quarter
df["QUARTER"] = df["FL_DATE"].dt.quarter

# Weekend
df["IS_WEEKEND"] = df["DAY_OF_WEEK"].isin(
    ["Saturday", "Sunday"]
).astype(int)

In [6]:
df[
    [
        "FL_DATE",
        "YEAR",
        "MONTH",
        "DAY",
        "DAY_OF_WEEK",
        "QUARTER",
        "IS_WEEKEND"
    ]
].head()

,FL_DATE,YEAR,MONTH,DAY,DAY_OF_WEEK,QUARTER,IS_WEEKEND
0,2019-01-09,2019,1,9,Wednesday,1,0
1,2022-11-19,2022,11,19,Saturday,4,1
2,2022-07-22,2022,7,22,Friday,3,0
3,2023-03-06,2023,3,6,Monday,1,0
4,2020-02-23,2020,2,23,Sunday,1,1


### 📝 Interpretation

Several calendar-based features were extracted from the flight date.

These variables may capture seasonal patterns, weekday effects, holiday travel trends, and operational differences between weekdays and weekends that influence flight delays.

In [7]:
df["DEP_HOUR"] = (
    df["CRS_DEP_TIME"] // 100
).astype(int)

In [8]:
df[["CRS_DEP_TIME", "DEP_HOUR"]].head(10)

,CRS_DEP_TIME,DEP_HOUR
0,1155,11
1,2120,21
2,954,9
3,1609,16
4,1840,18
5,1010,10
6,1010,10
7,1643,16
8,530,5
9,2125,21


### 📝 Interpretation

The scheduled departure hour was extracted from the scheduled departure time.

This feature allows the model to learn whether flights departing during certain hours of the day are more likely to experience delays.

# Time of Day

The scheduled departure hour is grouped into broader time-of-day categories.

These categories may capture operational patterns such as morning traffic peaks, afternoon congestion, and overnight operations, which can influence flight delays.

In [9]:
# Create Time of Day feature

def get_time_of_day(hour):
    if 5 <= hour < 12:
        return "Morning"
    elif 12 <= hour < 17:
        return "Afternoon"
    elif 17 <= hour < 21:
        return "Evening"
    else:
        return "Night"

df["TIME_OF_DAY"] = df["DEP_HOUR"].apply(get_time_of_day)

In [10]:
df[["DEP_HOUR", "TIME_OF_DAY"]].head(10)

,DEP_HOUR,TIME_OF_DAY
0,11,Morning
1,21,Night
2,9,Morning
3,16,Afternoon
4,18,Evening
5,10,Morning
6,10,Morning
7,16,Afternoon
8,5,Morning
9,21,Night


### 📝 Interpretation

The scheduled departure hour was transformed into a categorical feature representing the time of day.

This transformation reduces the number of unique values while preserving meaningful operational patterns that may influence flight delays.

# Season

Seasonal weather conditions and travel demand vary throughout the year.

A new feature representing the season is created from the flight month to capture these patterns.

In [11]:
# Create Season feature

def get_season(month):
    if month in [12, 1, 2]:
        return "Winter"
    elif month in [3, 4, 5]:
        return "Spring"
    elif month in [6, 7, 8]:
        return "Summer"
    else:
        return "Autumn"

df["SEASON"] = df["MONTH"].apply(get_season)

In [12]:
df[["MONTH", "SEASON"]].head(12)

,MONTH,SEASON
0,1,Winter
1,11,Autumn
2,7,Summer
3,3,Spring
4,2,Winter
5,7,Summer
6,6,Summer
7,7,Summer
8,2,Winter
9,8,Summer


### 📝 Interpretation

The flight month was grouped into four seasons.

Seasonal variation can influence flight operations due to weather conditions, passenger demand, and airport congestion.

# Distance Category

Flights are grouped into distance categories based on the total travel distance.

Categorizing flight distance may help the model identify differences between short-haul, medium-haul, and long-haul operations.

In [13]:
# Create Distance Category

def distance_category(distance):
    if distance < 500:
        return "Short"
    elif distance < 1500:
        return "Medium"
    else:
        return "Long"

df["DISTANCE_CATEGORY"] = df["DISTANCE"].apply(distance_category)

In [14]:
df[["DISTANCE", "DISTANCE_CATEGORY"]].head(10)

,DISTANCE,DISTANCE_CATEGORY
0,1065.0,Medium
1,1399.0,Medium
2,680.0,Medium
3,1589.0,Long
4,985.0,Medium
5,181.0,Short
6,399.0,Short
7,613.0,Medium
8,1379.0,Medium
9,1533.0,Long


### 📝 Interpretation

The continuous flight distance was converted into three operational categories.

This transformation allows the model to capture broad differences between short-, medium-, and long-haul flights while reducing sensitivity to small variations in distance.

# Route Feature

A new feature representing the flight route is created by combining the origin and destination airports.

Different routes may experience different weather conditions, traffic volumes, and airport operational characteristics, making this feature potentially valuable for prediction.

In [15]:
# Create Route

df["ROUTE"] = df["ORIGIN"] + "_" + df["DEST"]

In [16]:
df[["ORIGIN", "DEST", "ROUTE"]].head()

,ORIGIN,DEST,ROUTE
0,FLL,EWR,FLL_EWR
1,MSP,SEA,MSP_SEA
2,DEN,MSP,DEN_MSP
3,MSP,SFO,MSP_SFO
4,MCO,DFW,MCO_DFW


### 📝 Interpretation

A new route feature was created by combining the origin and destination airport codes.

This feature uniquely identifies each flight route and may help the model learn route-specific delay patterns.

# Data Leakage Analysis

Data leakage occurs when a machine learning model is trained using information that would not be available at the time a prediction is made.

The objective of this project is to predict whether a flight will experience a significant arrival delay **before the flight is completed**. Therefore, any variables generated during or after the flight must be excluded from the modeling dataset.

The following section identifies variables that introduce data leakage and explains why they cannot be used as predictors.

In [17]:
leakage_features = pd.DataFrame({
    "Feature": [
        "ARR_TIME",
        "ARR_DELAY",
        "WHEELS_ON",
        "TAXI_IN",
        "ELAPSED_TIME",
        "AIR_TIME",
        "DELAY_DUE_CARRIER",
        "DELAY_DUE_WEATHER",
        "DELAY_DUE_NAS",
        "DELAY_DUE_SECURITY",
        "DELAY_DUE_LATE_AIRCRAFT",
        "CANCELLATION_CODE"
    ],
    "Reason for Removal": [
        "Known only after arrival.",
        "Target variable used to create IS_DELAYED.",
        "Recorded after landing.",
        "Recorded after landing.",
        "Known after flight completion.",
        "Known after flight completion.",
        "Calculated after the delay occurs.",
        "Calculated after the delay occurs.",
        "Calculated after the delay occurs.",
        "Calculated after the delay occurs.",
        "Calculated after the delay occurs.",
        "Available only for cancelled flights."
    ]
})

leakage_features

,Feature,Reason for Removal
0,ARR_TIME,Known only after arrival.
1,ARR_DELAY,Target variable used to create IS_DELAYED.
2,WHEELS_ON,Recorded after landing.
3,TAXI_IN,Recorded after landing.
4,ELAPSED_TIME,Known after flight completion.
5,AIR_TIME,Known after flight completion.
6,DELAY_DUE_CARRIER,Calculated after the delay occurs.
7,DELAY_DUE_WEATHER,Calculated after the delay occurs.
8,DELAY_DUE_NAS,Calculated after the delay occurs.
9,DELAY_DUE_SECURITY,Calculated after the delay occurs.


### 📝 Interpretation

Several variables contain information that becomes available only after the flight has departed or arrived.

Including these variables during model training would allow the model to indirectly observe the outcome it is attempting to predict, resulting in overly optimistic performance and poor real-world generalization.

Therefore, these features will be removed before model development.

# Remove Leakage Features

The identified leakage variables are removed from the dataset to ensure that only information available before or at the scheduled departure time is used for prediction.

In [18]:
# Remove post-arrival, leakage, cancellation, and diversion columns

columns_to_drop = [
    "ARR_TIME",
    "ARR_DELAY",
    "WHEELS_ON",
    "TAXI_IN",
    "ELAPSED_TIME",
    "AIR_TIME",
    "TAXI_OUT",
    "WHEELS_OFF",
    "DELAY_DUE_CARRIER",
    "DELAY_DUE_WEATHER",
    "DELAY_DUE_NAS",
    "DELAY_DUE_SECURITY",
    "DELAY_DUE_LATE_AIRCRAFT",
    "CANCELLATION_CODE",
    "CANCELLED",
    "DIVERTED"
]

df_model = df.drop(
    columns=columns_to_drop,
    errors="raise"
)

print("Modeling dataset shape:")
print(df_model.shape)

print("\nRemaining columns:")
print(df_model.columns.tolist())

Modeling dataset shape:
(2913802, 28)

Remaining columns:
['FL_DATE', 'AIRLINE', 'AIRLINE_DOT', 'AIRLINE_CODE', 'DOT_CODE', 'FL_NUMBER', 'ORIGIN', 'ORIGIN_CITY', 'DEST', 'DEST_CITY', 'CRS_DEP_TIME', 'DEP_TIME', 'DEP_DELAY', 'CRS_ARR_TIME', 'CRS_ELAPSED_TIME', 'DISTANCE', 'IS_DELAYED', 'YEAR', 'MONTH', 'DAY', 'DAY_OF_WEEK', 'QUARTER', 'IS_WEEKEND', 'DEP_HOUR', 'TIME_OF_DAY', 'SEASON', 'DISTANCE_CATEGORY', 'ROUTE']


### 📝 Interpretation

The leakage features were successfully removed from the dataset.

The resulting dataset now contains only variables that are suitable for predictive modeling and can realistically be used to estimate flight delays before the flight is completed.

# Peak Travel Season

Certain months experience significantly higher passenger demand due to holidays and vacation periods.

A binary feature is created to identify flights occurring during peak travel seasons.

In [19]:
# Peak travel season

df_model["IS_PEAK_SEASON"] = (
    df_model["MONTH"].isin([6, 7, 8, 11, 12])
).astype(int)

df_model["IS_PEAK_SEASON"].value_counts()

IS_PEAK_SEASON
0    1680842
1    1232960
Name: count, dtype: int64

# Busy Departure Hours

Airport congestion tends to be higher during the morning and evening travel peaks.

A binary feature is created to identify flights scheduled during these busy operating periods.

In [20]:
# Busy departure hours

df_model["IS_BUSY_HOUR"] = (
    df_model["DEP_HOUR"].between(7, 10) |
    df_model["DEP_HOUR"].between(16, 19)
).astype(int)

df_model["IS_BUSY_HOUR"].value_counts()

IS_BUSY_HOUR
0    1474658
1    1439144
Name: count, dtype: int64

### 📝 Interpretation

Two additional binary features were created using aviation domain knowledge.

`IS_PEAK_SEASON` identifies flights during months with historically higher travel demand, while `IS_BUSY_HOUR` identifies departures scheduled during typical airport congestion periods.

These features may improve the model's ability to capture operational patterns associated with flight delays.

# Save the Engineered Dataset

The dataset is saved after completing feature engineering and removing leakage variables.

This version will be used for exploratory data analysis and model development.

In [21]:
engineered_output_path = (
    PROCESSED_DATA_DIRECTORY
    / "engineered_flight_data.csv"
)

df_model.to_csv(
    engineered_output_path,
    index=False
)

print("Engineered dataset saved to:")
print(engineered_output_path)

print("\nSaved dataset shape:")
print(df_model.shape)

print("\nSaved engineered columns:")
print(df_model.columns.tolist())

Engineered dataset saved to:
/Users/omarzakzook/Desktop/Flight_Delay_Prediction/data/processed/engineered_flight_data.csv

Saved dataset shape:
(2913802, 30)

Saved engineered columns:
['FL_DATE', 'AIRLINE', 'AIRLINE_DOT', 'AIRLINE_CODE', 'DOT_CODE', 'FL_NUMBER', 'ORIGIN', 'ORIGIN_CITY', 'DEST', 'DEST_CITY', 'CRS_DEP_TIME', 'DEP_TIME', 'DEP_DELAY', 'CRS_ARR_TIME', 'CRS_ELAPSED_TIME', 'DISTANCE', 'IS_DELAYED', 'YEAR', 'MONTH', 'DAY', 'DAY_OF_WEEK', 'QUARTER', 'IS_WEEKEND', 'DEP_HOUR', 'TIME_OF_DAY', 'SEASON', 'DISTANCE_CATEGORY', 'ROUTE', 'IS_PEAK_SEASON', 'IS_BUSY_HOUR']


# 📌 Notebook Summary

In this notebook, new predictive features were engineered from the original dataset.

The following tasks were completed:

- Extracted calendar-based features.
- Created departure time categories.
- Created seasonal features.
- Generated route and distance-based features.
- Added aviation domain knowledge features.
- Identified and removed data leakage variables.
- Saved the engineered dataset for subsequent analysis and model development.

The dataset is now ready for Exploratory Data Analysis (EDA), where feature relationships and patterns associated with flight delays will be investigated.